In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path
import numpy as np
import malaya_speech
from malaya_speech.model.clustering import StreamingKMeans

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
2025-09-28 14:18:38.046414: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759069118.055329   98962 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759069118.059697   98962 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1759069118.064886   98962 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:175906911

In [2]:
df = pd.read_parquet('Latin-Audio-audio.parquet')

In [3]:
mapping = {}
for i in tqdm(range(len(df))):
    mapping[df['audio'].iloc[i]] = i
len(mapping)

100%|██████████| 24299/24299 [00:00<00:00, 277278.71it/s]


24299

In [4]:
from datasets import load_dataset

ds = load_dataset("malaysia-ai/Multilingual-TTS", 'Latin-Audio')

Generating train split: 100%|██████████| 24299/24299 [00:00<00:00, 2954127.33 examples/s]


In [8]:
ds = ds['train'].to_pandas()

In [13]:
mapping[ds['audio_filename'].iloc[i]]

17772

In [19]:
import faiss

d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
data = {}
for i in tqdm(range(len(ds))):
    index_ = mapping[ds['audio_filename'].iloc[i]]
    v_f = f'Latin-Audio-audio/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[ds['audio_filename'].iloc[i]] = assign(v)
    except Exception as e:
        pass

100%|██████████| 24299/24299 [00:04<00:00, 5810.72it/s]


In [21]:
# data

In [28]:
rows = ds.to_dict(orient = 'records')
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [31]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

{'audio_filename': 'Latin-Audio_audio/Latin-Audio-part_0-audio-Seneca,_Medea_1-55_(Musa_Pedestris)_0000.mp3',
 'text': 'Dī coniugālīs, tūque geniali stōri.',
 'speaker': 'Latin-Audio_audio_0'}

In [32]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'Latin-Audio')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 76.66ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 2.02MB / 2.02MB,  206kB/s  
Processing Files (1 / 1): 100%|██████████| 2.02MB / 2.02MB,  202kB/s  
New Data Upload: 100%|██████████| 41.5kB / 41.5kB, 4.15kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:10<00:00, 10.52s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/5fb382953780d65b98d8715cc7d09374028c3689', commit_message='Upload dataset', commit_description='', oid='5fb382953780d65b98d8715cc7d09374028c3689', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)